# FMA Small Dataset Downloader

Downloads the [FMA Small](https://github.com/mdeff/fma) dataset (8,000 tracks of 30s, 8 genres, ~7.2 GiB) and its metadata.

- Songs are extracted to `data/fma_small/`
- Metadata is extracted to `data/fma_metadata/`
- Zips are cached in `temp/` — reused on re-runs

In [1]:
import os
import hashlib
import zipfile
import urllib.request
from pathlib import Path

In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT       = Path("data")
SONGS_DIR  = ROOT / "fma_small"
META_DIR   = ROOT / "fma_metadata"
TEMP_DIR   = Path("temp")

AUDIO_ZIP  = TEMP_DIR / "fma_small.zip"
META_ZIP   = TEMP_DIR / "fma_metadata.zip"

AUDIO_URL  = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
META_URL   = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"

# SHA-1 checksums from the official FMA README
AUDIO_SHA1 = "ade154f733639d52e35e32f5593efe5be76c6d70"
META_SHA1  = "f0df49ffe5f2a6008d7dc83c6915b31835dfe733"

# Expected number of MP3 tracks in fma_small
EXPECTED_TRACK_COUNT = 8000

In [3]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def sha1_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while buf := f.read(chunk):
            h.update(buf)
    return h.hexdigest()


def download(url: str, dest: Path) -> None:
    print(f"Downloading {url}")
    print(f"  → {dest}")

    def _progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(downloaded / total_size * 100, 100)
            gib = total_size / (1 << 30)
            print(f"\r  {pct:5.1f}%  ({gib:.2f} GiB total)", end="", flush=True)

    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print()  # newline after progress


def verify(path: Path, expected_sha1: str) -> bool:
    print(f"Verifying checksum for {path.name} …", end=" ", flush=True)
    digest = sha1_file(path)
    ok = digest == expected_sha1
    print("OK" if ok else f"MISMATCH (got {digest})")
    return ok


def count_mp3s(directory: Path) -> int:
    return sum(1 for _ in directory.rglob("*.mp3"))


def ensure_zip(zip_path: Path, url: str, sha1: str) -> None:
    """Download the zip only if not already cached, then verify."""
    if zip_path.exists():
        print(f"Found cached zip: {zip_path}")
        if not verify(zip_path, sha1):
            print("Checksum failed — re-downloading …")
            zip_path.unlink()
            download(url, zip_path)
            assert verify(zip_path, sha1), "Checksum still failed after re-download!"
    else:
        download(url, zip_path)
        assert verify(zip_path, sha1), "Checksum failed after download!"


def extract_zip(zip_path: Path, dest: Path) -> None:
    print(f"Extracting {zip_path.name} → {dest} …")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    print("Extraction complete.")

In [4]:
# ── Create directories ────────────────────────────────────────────────────────
ROOT.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)
print(f"Directories ready: {ROOT}/  {TEMP_DIR}/")

Directories ready: data/  temp/


In [ ]:
# ── Audio (fma_small) ─────────────────────────────────────────────────────────

mp3_count = count_mp3s(SONGS_DIR) if SONGS_DIR.exists() else 0
print(f"Songs already on disk: {mp3_count} / {EXPECTED_TRACK_COUNT}")

if mp3_count >= EXPECTED_TRACK_COUNT:
    print("All tracks present — skipping audio download & extraction.")
else:
    ensure_zip(AUDIO_ZIP, AUDIO_URL, AUDIO_SHA1)
    extract_zip(AUDIO_ZIP, ROOT)
    mp3_count = count_mp3s(SONGS_DIR)
    print(f"Tracks on disk after extraction: {mp3_count}")

Songs already on disk: 0 / 8000
  → temp/fma_small.zip
   16.1%  (7.15 GiB total)

In [ ]:
# ── Metadata (fma_metadata) ───────────────────────────────────────────────────

# Key files produced by the metadata archive
META_FILES = ["tracks.csv", "genres.csv", "features.csv", "echonest.csv"]
meta_present = all((META_DIR / f).exists() for f in META_FILES)

if meta_present:
    print("Metadata already present — skipping metadata download & extraction.")
else:
    ensure_zip(META_ZIP, META_URL, META_SHA1)
    extract_zip(META_ZIP, ROOT)
    print("Metadata files:", [f for f in META_FILES if (META_DIR / f).exists()])

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 50)
print(f"Songs dir  : {SONGS_DIR.resolve()}")
print(f"MP3 tracks : {count_mp3s(SONGS_DIR) if SONGS_DIR.exists() else 0}")
print(f"Metadata   : {META_DIR.resolve()}")
for f in META_FILES:
    p = META_DIR / f
    status = f"{p.stat().st_size / (1 << 20):.1f} MiB" if p.exists() else "MISSING"
    print(f"  {f:<20} {status}")
print("=" * 50)